# 第 3 章 · 自动体检

**为什么要做**：规矩靠人记，一定会忘。公司里让机器在每次存版本之前自动检查，不合格的存不进去。

**做完得到什么**：
- 每次存版本前，自动检查格式、低级错误、类型
- 有问题的代码存不进去
- 推送到 GitHub 之前，再自动跑一遍测试

从上往下一格一格点 ▶。

In [1]:
# 进到项目目录（这格不用看懂）
import os
import subprocess

os.chdir(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
os.environ.update(NO_COLOR="1", TERM="dumb", GIT_PAGER="cat", PAGER="cat",
                  GH_SPINNER_DISABLED="1", GH_PAGER="cat")
print("已进入项目目录")

已进入项目目录


## 1 · 有毛病的代码会被拦下

**为什么**：越早发现错误越便宜。在存版本的那一刻发现，比上线以后才发现便宜得多。

下面这格写一个有毛病的文件（把数字和文字相加），跑一遍"存版本前的检查"，然后把它删掉。

In [2]:
from pathlib import Path

bad = Path("src/raredx/bad.py")
bad.write_text('def add(a: int, b: int) -> int:\n    return a + "1"\n')
try:
    !uv run pre-commit run --files src/raredx/bad.py
finally:
    bad.unlink()
print("（有毛病的文件已删掉）")

trim trailing whitespace.................................................

Passed
fix end of files.........................................................

Passed
check yaml...........................................(no files to check)Skipped
check toml...........................................(no files to check)Skipped
check for merge conflicts................................................

Passed
check for added large files..............................................

Passed
detect private key.......................................................

Passed
ruff check...............................................................

Passed
ruff format..............................................................

Passed
mypy.....................................................................

Failed
- hook id: mypy
- exit code: 1

src/raredx/bad.py:2: error: Unsupported operand types for + ("int" and "str")  [operator]
Found 1 error in 1 file (checked 8 source files)



（有毛病的文件已删掉）


👀 看 **mypy … Failed** 那一行：它发现了"数字加文字"。只要有一项 Failed，这次存版本就会被拒绝。

## 2 · 正常的代码全部通过

**为什么**：检查要对所有代码都成立，不只是新写的那一点。

In [3]:
!uv run pre-commit run --all-files

trim trailing whitespace.................................................

Passed
fix end of files.........................................................

Passed
check yaml...............................................................

Passed
check toml...............................................................

Passed
check for merge conflicts................................................

...

Passed
detect private key.......................................................

Passed
ruff check...............................................................

Passed
ruff format..............................................................

Passed
mypy.....................................................................

Passed


👀 全部 Passed（Skipped 是"这一项没有要查的文件"）。我们项目里每一次存版本，都是这样过了检查才存进去的。

## 要记住的

| | 查什么 | 公司为什么要它 |
|---|---|---|
| ruff | 格式、低级错误 | 代码风格统一，低级错误不上线 |
| mypy | 类型对不对 | 运行之前就发现"数字加文字"这类错 |
| pytest | 跑测试 | 改了代码，确认原来的功能没坏 |
| pre-commit | 存版本前自动跑上面这些 | 规矩靠机器，不靠人记 |

**面试时可以这么说**："提交前用 pre-commit 自动跑 ruff 和 mypy，推送前跑 pytest。"